# Day 2 — Data quality and univariate analysis

This notebook profiles the approved cleaned dataset supplied by Darshan. It checks row and column counts, missingness, exact duplicates, repeated order IDs, key numeric summaries, and distributions of the main categorical fields. It does not impute values or change the source data.

`Delivery_Time` is delivery duration with an unconfirmed unit. `Pickup_Delay_Minutes` is order-to-pickup duration in minutes. The dataset has no promised delivery time, actual delivery timestamp, or delivery status, so promise-based delay and on-time/late/completion rates are unsupported.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path.home() / "Downloads" / "amazon_delivery_cleaned_updated.csv"
df = pd.read_csv(csv_path)
print(f"Source: {csv_path}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


In [ ]:
# Column names and data types
pd.DataFrame({"dtype": df.dtypes.astype(str), "non_missing": df.notna().sum(), "missing": df.isna().sum()})


In [ ]:
# Completeness, missingness and duplicate checks
quality = pd.DataFrame({
    "missing_records": df.isna().sum(),
    "missing_share": df.isna().mean().mul(100).round(2),
})
quality = quality.loc[quality["missing_records"].gt(0)].sort_values("missing_records", ascending=False)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate Order_ID values:", int(df["Order_ID"].duplicated().sum()))
display(quality)


In [ ]:
# Check the supplied missingness indicators against the fields they describe
indicator_checks = pd.DataFrame({
    "check": ["Store_Location_Missing equals missing store coordinates", "Order_Time_Missing equals missing Order_Time"],
    "matches": [
        bool((df["Store_Location_Missing"].eq(1) == df[["Store_Latitude", "Store_Longitude"]].isna().all(axis=1)).all()),
        bool((df["Order_Time_Missing"].eq(1) == df["Order_Time"].isna()).all()),
    ],
})
display(indicator_checks)


In [ ]:
# Numeric distributions for the main numeric fields
numeric_fields = ["Agent_Age", "Agent_Rating", "Delivery_Time", "Pickup_Delay_Minutes"]
numeric_summary = df[numeric_fields].describe(percentiles=[.25, .5, .75, .9]).T
numeric_summary.insert(1, "missing", df[numeric_fields].isna().sum())
display(numeric_summary.round(2))


In [ ]:
# Histograms; the Delivery_Time axis intentionally has no unit label
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, field, title, xlabel in [
    (axes[0, 0], "Agent_Age", "Agent age", "Age (unit as recorded)"),
    (axes[0, 1], "Agent_Rating", "Agent rating", "Rating (scale as recorded)"),
    (axes[1, 0], "Delivery_Time", "Delivery duration", "Delivery_Time (unit unconfirmed)"),
    (axes[1, 1], "Pickup_Delay_Minutes", "Order-to-pickup duration", "Minutes"),
]:
    df[field].dropna().plot.hist(bins=25, ax=ax, edgecolor="white")
    ax.set(title=title, xlabel=xlabel, ylabel="Records")
fig.tight_layout()
plt.show()


In [ ]:
# Counts and shares for the main categorical fields
categorical_fields = ["Weather", "Traffic", "Vehicle", "Area", "Order_Time_Period", "Category"]
for field in categorical_fields:
    counts = df[field].value_counts(dropna=False).rename_axis("value").to_frame("records")
    counts["share_percent"] = (counts["records"] / len(df) * 100).round(2)
    print(f"\n{field}")
    display(counts)


In [ ]:
# Categorical distribution charts; missing values are shown explicitly
plot_fields = ["Weather", "Traffic", "Vehicle", "Area", "Order_Time_Period", "Category"]
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for ax, field in zip(axes.flat, plot_fields):
    counts = df[field].fillna("Missing").value_counts()
    counts.sort_values().plot.barh(ax=ax)
    ax.set(title=field, xlabel="Records", ylabel="")
fig.tight_layout()
plt.show()


## Notes for interpretation

- Darshan said the 91 missing order-time/weather/traffic records were retained because reliable replacements were unavailable.
- Darshan said store coordinates originally coded as `(0, 0)` were converted to missing; 3,505 rows have both store coordinates missing.
- `Delivery_Time` is delivery duration; the unit is not verified. Do not label it in minutes.
- The data lacks promised delivery time, actual delivery timestamp, and delivery status. Do not calculate on-time rate, late rate, completion rate, or promise-based delay.
- `Pickup_Delay_Minutes` is order-to-pickup duration in minutes, not delivery duration.
- Counts and distributions describe this file only and do not establish causal relationships.
